<a href="https://colab.research.google.com/github/juniti-y/RLseminar/blob/main/GymDemo01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# gymnasiumパッケージのインストール

In [1]:
!pip install -q gymnasium
!pip install -q swig
!pip install -q "stable-baselines3[extra]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 13.8 MB/s eta 0:00:00


## インストールが正常に完了したかを確認

In [2]:
import gymnasium as gym

# CartPole環境オブジェクトを作成
env = gym.make("CartPole-v1")

# 環境を初期状態にリセット
observation, info = env.reset()

# いくつかのステップを実行
for _ in range(10):
    # actionをランダムサンプリング
    action = env.action_space.sample()
    # 状態を次状態に遷移し、観測値、報酬、終了状態に到達したかを取得
    observation, reward, terminated, truncated, info = env.step(action)
    # 終了状態に到達したら終了
    if terminated or truncated:
        observation, info = env.reset()

# 環境オブジェクトをクローズ
env.close()
print("Successfully finished")


Successfully finished


# 環境から基本情報を得る

## 観測空間を取得


In [3]:
# 環境オブジェクトを作成（CartPoleを例に）
env = gym.make("CartPole-v1")
# 観測空間を表示
print(env.observation_space)

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)


## 行動空間を取得

In [4]:
print(env.action_space)

Discrete(2)


## タスク実行状態の可視化
（Pendulumタスクを例に）

In [5]:
import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
# env = gym.make("CartPole-v1", render_mode="rgb_array")
env = gym.make("Pendulum-v1", render_mode="rgb_array")
trigger = lambda t: t % 10 == 0 #動画を保存をするエピソードの指定
env = gym.wrappers.RecordVideo(env, video_folder="./", episode_trigger=trigger, disable_logger=True)
# 50エピソードランダムアクションで実行し、動画を保存
for i in range(50):
  env.reset(seed=123)
  termination, truncation = False, False
  while not (termination or truncation):
    obs, rew, termination, truncation, info = env.step(env.action_space.sample()) #policyはランダム
env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


実行後に、「rl-video-episode-0.mp4」のようなファイルが作成されているので、それを動画プレーヤで再生する。

# 公開ライブラリ（stable_baselines3）の強化学習モデルを使って学習する

## 必要なモジュールの呼び出し

In [18]:
import gymnasium as gym
from stable_baselines3 import PPO
from gymnasium.wrappers import RecordVideo

## 環境を作成（CartPoleを例に）

In [12]:
env = gym.make("CartPole-v1")

## 学習エージェント（PPOエージェント）を作成

In [13]:
model = PPO(policy="MlpPolicy", env=env,verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## 学習を実行する

In [14]:
model.learn(total_timesteps=1000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.6     |
|    ep_rew_mean     | 23.6     |
| time/              |          |
|    fps             | 1273     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------


## 学習済みモデルを保存する

In [15]:
model.save("ppo_cartpole")

## 環境を一旦クローズして、学習フェーズを終了する

In [16]:
env.close()

## テストフェーズ（学習済みモデルを実際に環境で動かす）

In [19]:
# 動画作成モードで環境を作成
env = gym.make("CartPole-v1", render_mode="rgb_array")
env = gym.wrappers.RecordVideo(env, video_folder="./", disable_logger=True, name_prefix="ppo_cartpole_video")

# 保存した学習済みモデルをロード
model = PPO.load("ppo_cartpole", env=env)

# 環境を初期状態にリセット
obs, info = env.reset()
# 1試行分を動画記録付きで実行
for step in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


# 自由課題：タスクを変更したり、学習モデルを変更して学習の様子を観察してみる
（下記にCartporl課題+PPO学習モデルのフルコードを提示するのでそれを編集するところから始めると良い）

In [20]:
#############################################
# 定数の定義
#############################################
task_name = "CartPole-v1"
policy_model = "MlpPolicy"
model_file = "ppo_cartpole"
video_file_prefix = "ppo_cartpole_video"

#############################################
# モジュールの読み込み
#############################################

import gymnasium as gym
from stable_baselines3 import PPO
from gymnasium.wrappers import RecordVideo

#############################################
# 環境を作る
#############################################
env = gym.make(task_name)

#############################################
# 学習エージェント（PPOエージェント）を作成
#############################################

model = PPO(policy=policy_model, env=env,verbose=1)

#############################################
# 学習の実行とモデルの保存
#############################################

model.learn(total_timesteps=1000)
model.save(model_file)
env.close()

#############################################
# テストフェーズの実行
#############################################

# 動画作成モードで環境を作成
env = gym.make(task_name, render_mode="rgb_array")
env = gym.wrappers.RecordVideo(env, video_folder="./", disable_logger=True, name_prefix=video_file_prefix)

# 保存した学習済みモデルをロード
model = PPO.load(model_file, env=env)

# 環境を初期状態にリセット
obs, info = env.reset()
# 1試行分を動画記録付きで実行
for step in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        obs, info = env.reset()
env.close()


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.3     |
|    ep_rew_mean     | 22.3     |
| time/              |          |
|    fps             | 1002     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## 参考
Gymnasiumの代表的な環境としては以下のようなものがある

*   MountainCar: "MountainCar-v0"
*   Pendulum: "Pendulum-v1"
*   Acrobot: "Acrobot-v1"

Stable-Baselines3 の代表的モデルとしては以下のようなものがある
